In [8]:
import os
import sys
import time
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv

from langchain_openai import (
    ChatOpenAI,
    OpenAIEmbeddings
)

from langchain_community.vectorstores import FAISS


load_dotenv()

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


from Agents.query_agent import QueryAgent
from Agents.retrieval_agent import RetrievalAgent
from Agents.review_agent import ReviewAgent
from Agents.ranking_agent import RankingAgent


llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


reviews_path = (
    PROJECT_ROOT
    / "Data"
    / "Cleaned"
    / "reviews_sample.parquet"
)

reviews_df = pd.read_parquet(
    reviews_path
)


faiss_path = (
    PROJECT_ROOT
    / "Data"
    / "Cleaned"
    / "faiss_index"
)

vector_db = FAISS.load_local(
    folder_path=str(faiss_path),
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)

print(
    "Resources loaded successfully."
)

Resources loaded successfully.


In [9]:
query_agent = QueryAgent()

retrieval_agent = RetrievalAgent(
    vector_db=vector_db
)

review_agent = ReviewAgent(
    llm=llm,
    reviews_df=reviews_df
)

ranking_agent = RankingAgent(
    llm=llm,
    explanation_limit=2,
    use_llm_explanations=True
)

print(
    "Four independent agents initialized."
)

Four independent agents initialized.


In [10]:
def safe_float(
    value,
    default=0.0
):
    try:
        if value is None or pd.isna(value):
            return default

        return float(value)

    except (TypeError, ValueError):
        return default


def safe_int(
    value,
    default=0
):
    try:
        if value is None or pd.isna(value):
            return default

        return int(float(value))

    except (TypeError, ValueError):
        return default

In [11]:
def enrich_products_without_coordination(
    retrieved_products: list,
    max_reviews: int = 3
) -> list[dict]:

    enriched_products = []

    for document, retrieval_score in retrieved_products:

        metadata = document.metadata.copy()

        parent_asin = metadata.get(
            "parent_asin"
        )

        if not parent_asin:
            continue

        try:

            review_result = (
                review_agent.summarize_reviews(
                    parent_asin=str(parent_asin),
                    max_reviews=max_reviews
                )
            )

            if hasattr(
                review_result,
                "model_dump"
            ):
                review_result = (
                    review_result.model_dump()
                )

            elif not isinstance(
                review_result,
                dict
            ):
                review_result = {}

        except Exception:

            review_result = {}

        enriched_product = {
            "parent_asin": str(
                parent_asin
            ),

            "title": str(
                metadata.get(
                    "title",
                    review_result.get(
                        "title",
                        "Unknown Product"
                    )
                )
            ),

            "brand": metadata.get(
                "brand"
            ),

            "product_type": metadata.get(
                "product_type"
            ),

            "price": metadata.get(
                "price"
            ),

            "categories": metadata.get(
                "categories",
                []
            ),

            "product_text": (
                document.page_content
            ),

            "retrieval_score": safe_float(
                metadata.get(
                    "retrieval_score",
                    retrieval_score
                )
            ),

            "similarity": safe_float(
                metadata.get(
                    "similarity",
                    0
                )
            ),

            "average_rating": safe_float(
                metadata.get(
                    "average_rating",
                    review_result.get(
                        "average_rating",
                        0
                    )
                )
            ),

            "rating_number": safe_float(
                metadata.get(
                    "rating_number",
                    0
                )
            ),

            "review_count": safe_int(
                review_result.get(
                    "review_count",
                    0
                )
            ),

            "pros": review_result.get(
                "pros",
                []
            ) or [],

            "cons": review_result.get(
                "cons",
                []
            ) or [],

            "overall_sentiment": (
                review_result.get(
                    "overall_sentiment",
                    "Unknown"
                )
            ),

            "recommended_for": (
                review_result.get(
                    "recommended_for",
                    ""
                )
            ),

            "avoid_if": review_result.get(
                "avoid_if",
                ""
            ),

            "review_summary": (
                review_result.get(
                    "summary",
                    ""
                )
            ),

            "sample_reviews": (
                review_result.get(
                    "sample_reviews",
                    ""
                )
            )
        }

        enriched_products.append(
            enriched_product
        )

    return enriched_products

In [12]:
def run_multi_agent_without_coordination(
    user_query: str,
    retrieval_k: int = 10,
    max_reviews: int = 3,
    verbose: bool = True
) -> dict:

    if not user_query or not user_query.strip():
        raise ValueError(
            "User query cannot be empty."
        )

    pipeline_start = time.perf_counter()

    # -----------------------------------------
    # Query Agent
    # -----------------------------------------

    query_start = time.perf_counter()

    structured_query = query_agent.parse(
        user_query
    )

    query_latency = (
        time.perf_counter()
        - query_start
    )

    # -----------------------------------------
    # Retrieval Agent
    # -----------------------------------------

    retrieval_start = time.perf_counter()

    retrieved_products = (
        retrieval_agent.retrieve(
            query=structured_query,
            k=retrieval_k
        )
    )

    retrieval_latency = (
        time.perf_counter()
        - retrieval_start
    )

    if not retrieved_products:

        total_latency = (
            time.perf_counter()
            - pipeline_start
        )

        return {
            "system": (
                "multi_agent_without_coordination"
            ),
            "user_query": user_query,
            "structured_query": structured_query,
            "retrieved_products": [],
            "enriched_products": [],
            "ranking_output": None,
            "recommended_product": None,
            "recommended_asin": None,
            "latency": {
                "query_agent": round(
                    query_latency,
                    4
                ),
                "retrieval_agent": round(
                    retrieval_latency,
                    4
                ),
                "review_agent": 0,
                "ranking_agent": 0,
                "total": round(
                    total_latency,
                    4
                )
            },
            "error": (
                "No products were retrieved."
            )
        }

    # -----------------------------------------
    # Review Agent
    # -----------------------------------------

    review_start = time.perf_counter()

    enriched_products = (
        enrich_products_without_coordination(
            retrieved_products=(
                retrieved_products
            ),
            max_reviews=max_reviews
        )
    )

    review_latency = (
        time.perf_counter()
        - review_start
    )

    if not enriched_products:

        total_latency = (
            time.perf_counter()
            - pipeline_start
        )

        return {
            "system": (
                "multi_agent_without_coordination"
            ),
            "user_query": user_query,
            "structured_query": structured_query,
            "retrieved_products": (
                retrieved_products
            ),
            "enriched_products": [],
            "ranking_output": None,
            "recommended_product": None,
            "recommended_asin": None,
            "latency": {
                "query_agent": round(
                    query_latency,
                    4
                ),
                "retrieval_agent": round(
                    retrieval_latency,
                    4
                ),
                "review_agent": round(
                    review_latency,
                    4
                ),
                "ranking_agent": 0,
                "total": round(
                    total_latency,
                    4
                )
            },
            "error": (
                "No products were available "
                "after review enrichment."
            )
        }

    # -----------------------------------------
    # Ranking Agent
    # -----------------------------------------

    ranking_start = time.perf_counter()

    ranking_output = (
        ranking_agent.rank_products(
            query=structured_query,
            products=enriched_products
        )
    )

    ranking_latency = (
        time.perf_counter()
        - ranking_start
    )

    ranked_products = (
        ranking_output.products
        if ranking_output is not None
        else []
    )

    top_product = (
        ranked_products[0]
        if ranked_products
        else None
    )

    total_latency = (
        time.perf_counter()
        - pipeline_start
    )

    if verbose:

        print("=" * 100)
        print(
            "MULTI-AGENT WITHOUT COORDINATION"
        )
        print("=" * 100)

        print(
            "Structured query:",
            structured_query
        )

        print(
            "Retrieved products:",
            len(retrieved_products)
        )

        print(
            "Enriched products:",
            len(enriched_products)
        )

        print()

        for product in ranked_products:

            print(
                f"{product.rank}. "
                f"{product.title} "
                f"| Score: "
                f"{product.final_score:.4f}"
            )

            print(
                "Reason:",
                product.reason
            )

            print()

        print(
            "Recommended product:",
            (
                top_product.title
                if top_product
                else None
            )
        )

        print(
            "Total latency:",
            round(
                total_latency,
                2
            ),
            "seconds"
        )

    return {
        "system": (
            "multi_agent_without_coordination"
        ),
        "user_query": user_query,
        "structured_query": structured_query,
        "retrieved_products": (
            retrieved_products
        ),
        "enriched_products": (
            enriched_products
        ),
        "ranking_output": (
            ranking_output
        ),
        "recommended_product": (
            top_product.title
            if top_product
            else None
        ),
        "recommended_asin": (
            top_product.parent_asin
            if top_product
            else None
        ),
        "latency": {
            "query_agent": round(
                query_latency,
                4
            ),
            "retrieval_agent": round(
                retrieval_latency,
                4
            ),
            "review_agent": round(
                review_latency,
                4
            ),
            "ranking_agent": round(
                ranking_latency,
                4
            ),
            "total": round(
                total_latency,
                4
            )
        },
        "error": None
    }

In [13]:
# =====================================================
# FINAL MULTI-AGENT WITHOUT COORDINATION RUN
# =====================================================

from langchain_community.callbacks.manager import (
    get_openai_callback
)

import pandas as pd


FINAL_FOLDER = (
    PROJECT_ROOT
    / "Results"
    / "Final"
)


benchmark_df = pd.read_csv(
    FINAL_FOLDER
    / "final_60_query_benchmark.csv"
)


no_coord_final_rows = []


for _, test_case in benchmark_df.iterrows():

    test_id = int(
        test_case["test_id"]
    )

    user_query = str(
        test_case["user_query"]
    )

    print(
        "No Coordination:",
        test_id
    )


    try:

        with get_openai_callback() as cb:

            result = (
                run_multi_agent_without_coordination(
                    user_query=user_query,
                    retrieval_k=10,
                    max_reviews=3,
                    verbose=False
                )
            )


        ranking_output = result.get(
            "ranking_output"
        )


        ranked_products = (
            ranking_output.products
            if ranking_output is not None
            else []
        )


        top_5_asins = [
            product.parent_asin
            for product
            in ranked_products[:5]
            if product.parent_asin
        ]


        top_5_titles = [
            product.title
            for product
            in ranked_products[:5]
            if product.title
        ]


        no_coord_final_rows.append({

            "test_id":
                test_id,

            "category":
                test_case["category"],

            "user_query":
                user_query,

            "recommended_product":
                result.get(
                    "recommended_product"
                ),

            "recommended_asin":
                result.get(
                    "recommended_asin"
                ),

            "top_5_asins":
                "|".join(
                    top_5_asins
                ),

            "top_5_titles":
                " || ".join(
                    top_5_titles
                ),

            "total_latency":
                result[
                    "latency"
                ][
                    "total"
                ],

            "prompt_tokens":
                int(
                    cb.prompt_tokens
                ),

            "completion_tokens":
                int(
                    cb.completion_tokens
                ),

            "total_tokens":
                int(
                    cb.total_tokens
                ),

            "estimated_cost_usd":
                float(
                    cb.total_cost
                ),

            "error":
                result.get(
                    "error"
                )
        })


    except Exception as error:

        no_coord_final_rows.append({

            "test_id":
                test_id,

            "category":
                test_case["category"],

            "user_query":
                user_query,

            "recommended_product":
                None,

            "recommended_asin":
                None,

            "top_5_asins":
                "",

            "top_5_titles":
                "",

            "total_latency":
                0,

            "prompt_tokens":
                0,

            "completion_tokens":
                0,

            "total_tokens":
                0,

            "estimated_cost_usd":
                0,

            "error":
                str(error)
        })


no_coord_final_df = pd.DataFrame(
    no_coord_final_rows
)


display(
    no_coord_final_df
)


no_coord_final_df.to_csv(
    FINAL_FOLDER
    / "final_no_coordination_results.csv",
    index=False
)


print(
    "Final No-Coordination run complete."
)

No Coordination: 1
No Coordination: 2
No Coordination: 3
No Coordination: 4
No Coordination: 5
No Coordination: 6
No Coordination: 7
No Coordination: 8
No Coordination: 9
No Coordination: 10
No Coordination: 11
No Coordination: 12
No Coordination: 13
No Coordination: 14
No Coordination: 15
No Coordination: 16
No Coordination: 17
No Coordination: 18
No Coordination: 19
No Coordination: 20
No Coordination: 21
No Coordination: 22
No Coordination: 23
No Coordination: 24
No Coordination: 25
No Coordination: 26
No Coordination: 27
No Coordination: 28
No Coordination: 29
No Coordination: 30
No Coordination: 31
No Coordination: 32
No Coordination: 33
No Coordination: 34
No Coordination: 35
No Coordination: 36
No Coordination: 37
No Coordination: 38
No Coordination: 39
No Coordination: 40
No Coordination: 41
No Coordination: 42
No Coordination: 43
No Coordination: 44
No Coordination: 45
No Coordination: 46
No Coordination: 47
No Coordination: 48
No Coordination: 49
No Coordination: 50
No Coordi

,test_id,category,user_query,recommended_product,recommended_asin,top_5_asins,top_5_titles,total_latency,prompt_tokens,completion_tokens,total_tokens,estimated_cost_usd,error
0,1,Brand Only,Recommend a Samsung phone.,Samsung Galaxy S5 SM-G900H Factory Unlocked Ce...,B00JKSUHLU,B00JKSUHLU|B00D93LOY6|B00LMJDP1Y|B00CGIULGC|B0...,Samsung Galaxy S5 SM-G900H Factory Unlocked Ce...,20.1184,3705,810,4515,0.001042,None
1,2,Brand Only,Recommend an Apple phone.,Apple iPhone 5 - 16GB (Black) Factory Unlocked,B00CX0OZHY,B00CX0OZHY,Apple iPhone 5 - 16GB (Black) Factory Unlocked,3.4303,1238,100,1338,0.000246,None
2,3,Brand Only,Suggest a Motorola smartphone.,Heart with Stars (Muti Color) Cell Phone Charm...,B004NULVAQ,B004NULVAQ|B001VF7LWI|B002VRO83K|B006P82YC8|B0...,Heart with Stars (Muti Color) Cell Phone Charm...,11.3453,2695,550,3245,0.000734,None
3,4,Brand Only,Recommend a Nokia phone.,Nokia E5-00 Unlocked GSM Phone with Easy Email...,B003X26SLM,B003X26SLM|B0055JQA46|B00UYAE1K6|B0014DGPZG,Nokia E5-00 Unlocked GSM Phone with Easy Email...,8.5073,2330,448,2778,0.000618,None
4,5,Brand Only,Suggest a Google phone.,"Google Pixel XL 128GB - 5.5"" Android GSM 4G LT...",B01M27MVQI,B01M27MVQI|B00INKCR14|B08BXBT8MD,"Google Pixel XL 128GB - 5.5"" Android GSM 4G LT...",6.7497,2045,321,2366,0.000499,None
5,6,Brand + Feature,Recommend a Samsung phone with a good camera.,Samsung Galaxy Mega 6.3 I9200 8GB Unlocked GSM...,B00D93LOY6,B00D93LOY6|B09C6N8P6Y|B00CGIULGC|B0B1QXHB7L|B0...,Samsung Galaxy Mega 6.3 I9200 8GB Unlocked GSM...,9.1145,2320,413,2733,0.000596,None
6,7,Brand + Feature,Recommend a Samsung phone with long battery life.,"SAMSUNG Galaxy S21 Ultra 5G, 128GB, Phantom Bl...",B09S6VKCLX,B09S6VKCLX|B09WT8N5X7|B09X9FC88M|B00JKSUHLU|B0...,"SAMSUNG Galaxy S21 Ultra 5G, 128GB, Phantom Bl...",6.1771,1798,282,2080,0.000439,None
7,8,Brand + Feature,Recommend an Apple phone with a good camera.,SteadyMate SM1 HD Professional Handheld Camera...,B00ORZVW42,B00ORZVW42,SteadyMate SM1 HD Professional Handheld Camera...,3.0321,1268,79,1347,0.000238,None
8,9,Brand + Feature,Suggest a Motorola phone with a good camera.,Motorola DROID A855 Android Phone (Verizon Wir...,B002VRO83K,B002VRO83K|B006P82YC8|B001COKAT4|B004P551BE|B0...,Motorola DROID A855 Android Phone (Verizon Wir...,2.9785,1403,137,1540,0.000293,None
9,10,Brand + Feature,Recommend a Samsung phone with fast performance.,Samsung Galaxy S5 SM-G900H Factory Unlocked Ce...,B00JKSUHLU,B00JKSUHLU|B00D93LOY6|B00LMJDP1Y|B09C6N8P6Y|B0...,Samsung Galaxy S5 SM-G900H Factory Unlocked Ce...,2.9848,1547,143,1690,0.000318,None


Final No-Coordination run complete.


In [14]:
# =====================================================
# TOKEN TRACKING - MULTI-AGENT WITHOUT COORDINATION
# =====================================================

from langchain_community.callbacks.manager import get_openai_callback
import pandas as pd


# Load same 15 test queries
test_queries_df = pd.read_csv(
    PROJECT_ROOT
    / "Results"
    / "coordinated_pipeline_test_results.csv"
)[["test_id", "user_query"]].drop_duplicates(
    subset=["test_id"]
).sort_values("test_id")


no_coord_token_rows = []

print("Queries loaded:", len(test_queries_df))


for _, row in test_queries_df.iterrows():

    test_id = int(row["test_id"])
    user_query = str(row["user_query"])

    print(
        "Running Multi-Agent Without Coordination:",
        test_id
    )

    with get_openai_callback() as cb:

        result = run_multi_agent_without_coordination(
            user_query=user_query,
            retrieval_k=10,
            max_reviews=3,
            verbose=False
        )

    no_coord_token_rows.append({
        "test_id": test_id,
        "system": "Multi-Agent Without Coordination",
        "user_query": user_query,
        "prompt_tokens": int(cb.prompt_tokens),
        "completion_tokens": int(cb.completion_tokens),
        "total_tokens": int(cb.total_tokens),
        "estimated_cost_usd": float(cb.total_cost)
    })


no_coord_token_df = pd.DataFrame(
    no_coord_token_rows
)


display(
    no_coord_token_df
)


# Save results
no_coord_token_df.to_csv(
    PROJECT_ROOT
    / "Results"
    / "no_coord_token_usage.csv",
    index=False
)


print(
    "Saved Multi-Agent Without Coordination token results."
)

Queries loaded: 15
Running Multi-Agent Without Coordination: 1
Running Multi-Agent Without Coordination: 2
Running Multi-Agent Without Coordination: 3
Running Multi-Agent Without Coordination: 4
Running Multi-Agent Without Coordination: 5
Running Multi-Agent Without Coordination: 6
Running Multi-Agent Without Coordination: 7
Running Multi-Agent Without Coordination: 8
Running Multi-Agent Without Coordination: 9
Running Multi-Agent Without Coordination: 10
Running Multi-Agent Without Coordination: 11
Running Multi-Agent Without Coordination: 12
Running Multi-Agent Without Coordination: 13
Running Multi-Agent Without Coordination: 14
Running Multi-Agent Without Coordination: 15


,test_id,system,user_query,prompt_tokens,completion_tokens,total_tokens,estimated_cost_usd
0,1,Multi-Agent Without Coordination,Recommend a Samsung phone with a good camera a...,1476,168,1644,0.000322
1,2,Multi-Agent Without Coordination,I want an Apple phone with excellent camera qu...,1270,84,1354,0.000241
2,3,Multi-Agent Without Coordination,Suggest a Google phone with fast performance.,1540,124,1664,0.000305
3,4,Multi-Agent Without Coordination,Recommend a Samsung phone under 300 with a goo...,1567,155,1722,0.000328
4,5,Multi-Agent Without Coordination,Recommend a smartphone under 250 with long bat...,1525,149,1674,0.000318
5,6,Multi-Agent Without Coordination,Recommend a phone with a great camera.,1572,136,1708,0.000317
6,7,Multi-Agent Without Coordination,Recommend a phone with long battery life.,1540,139,1679,0.000314
7,8,Multi-Agent Without Coordination,Suggest a smartphone for gaming.,1534,142,1676,0.000315
8,9,Multi-Agent Without Coordination,"Recommend a Samsung phone with a good camera, ...",1567,144,1711,0.000321
9,10,Multi-Agent Without Coordination,Recommend an Apple phone with a good camera an...,1715,240,1955,0.000401


Saved Multi-Agent Without Coordination token results.


In [15]:
test_result = (
    run_multi_agent_without_coordination(
        user_query=(
            "Recommend a Samsung phone with "
            "a good camera and long battery life."
        ),
        retrieval_k=10,
        max_reviews=3,
        verbose=True
    )
)

MULTI-AGENT WITHOUT COORDINATION
Structured query: product_type='smartphone' brand='Samsung' budget=None features=['good camera', 'long battery life', 'camera']
Retrieved products: 8
Enriched products: 8

1. Samsung Galaxy S21 FE 5G Cell Phone, Factory Unlocked Android Smartphone, 128GB, 120Hz Display, Pro Grade Camera, All Day Intelligent Battery, (Olive Green) (Renewed) | Score: 0.6805
Reason: The Samsung Galaxy S21 FE 5G is ranked first due to its mention of a 'Pro Grade Camera' and 'All Day Intelligent Battery' in the product evidence, aligning with the requested features of a good camera and long battery life. No customer-review evidence was available.

2. SAMSUNG Galaxy S21 Ultra 5G, 128GB, Phantom Black - Unlocked (Renewed Premium) | Score: 0.6680
Reason: The SAMSUNG Galaxy S21 Ultra 5G is ranked second as it is described to have a battery that 'goes all day and then some,' supporting the requested feature of long battery life. However, no customer-review evidence was available.

In [16]:
test_result["structured_query"]

ProductQuery(product_type='smartphone', brand='Samsung', budget=None, features=['good camera', 'long battery life', 'camera'])

In [17]:
test_result["ranking_output"]

RankingOutput(products=[RankedProduct(rank=1, parent_asin='B0B1QXHB7L', title='Samsung Galaxy S21 FE 5G Cell Phone, Factory Unlocked Android Smartphone, 128GB, 120Hz Display, Pro Grade Camera, All Day Intelligent Battery, (Olive Green) (Renewed)', average_rating=4.6, retrieval_score=0.6121030579493104, final_score=0.680517, reason="The Samsung Galaxy S21 FE 5G is ranked first due to its mention of a 'Pro Grade Camera' and 'All Day Intelligent Battery' in the product evidence, aligning with the requested features of a good camera and long battery life. No customer-review evidence was available."), RankedProduct(rank=2, parent_asin='B09S6VKCLX', title='SAMSUNG Galaxy S21 Ultra 5G, 128GB, Phantom Black - Unlocked (Renewed Premium)', average_rating=4.2, retrieval_score=0.6048606987987102, final_score=0.668026, reason="The SAMSUNG Galaxy S21 Ultra 5G is ranked second as it is described to have a battery that 'goes all day and then some,' supporting the requested feature of long battery life

In [18]:
test_result["latency"]

{'query_agent': 1.196,
 'retrieval_agent': 0.337,
 'review_agent': 0.0001,
 'ranking_agent': 2.0454,
 'total': 3.5785}

### TESTING

In [19]:
# =====================================================
# TEST QUERIES FOR SINGLE-AGENT PIPELINE
# =====================================================

test_queries = [

    # -----------------------------------------
    # Brand + Feature Queries
    # -----------------------------------------

    {
        "test_id": 1,
        "category": "Brand + Features",
        "query": (
            "Recommend a Samsung phone with a good camera "
            "and long battery life."
        )
    },

    {
        "test_id": 2,
        "category": "Brand + Features",
        "query": (
            "I want an Apple phone with excellent camera quality."
        )
    },

    {
        "test_id": 3,
        "category": "Brand + Features",
        "query": (
            "Suggest a Google phone with fast performance."
        )
    },

    # -----------------------------------------
    # Budget Queries
    # -----------------------------------------

    {
        "test_id": 4,
        "category": "Budget",
        "query": (
            "Recommend a Samsung phone under 300 "
            "with a good camera."
        )
    },

    {
        "test_id": 5,
        "category": "Budget",
        "query": (
            "Recommend a smartphone under 250 "
            "with long battery life."
        )
    },

    # -----------------------------------------
    # Feature Only Queries
    # -----------------------------------------

    {
        "test_id": 6,
        "category": "Feature Only",
        "query": (
            "Recommend a phone with a great camera."
        )
    },

    {
        "test_id": 7,
        "category": "Feature Only",
        "query": (
            "Recommend a phone with long battery life."
        )
    },

    {
        "test_id": 8,
        "category": "Feature Only",
        "query": (
            "Suggest a smartphone for gaming."
        )
    },

    # -----------------------------------------
    # Multiple Feature Queries
    # -----------------------------------------

    {
        "test_id": 9,
        "category": "Multiple Features",
        "query": (
            "Recommend a Samsung phone with a good camera, "
            "long battery life and fast performance."
        )
    },

    {
        "test_id": 10,
        "category": "Multiple Features",
        "query": (
            "Recommend an Apple phone with a good camera "
            "and large storage."
        )
    },

    # -----------------------------------------
    # Ambiguous Queries
    # -----------------------------------------

    {
        "test_id": 11,
        "category": "Ambiguous",
        "query": (
            "Recommend the best Samsung phone."
        )
    },

    {
        "test_id": 12,
        "category": "Ambiguous",
        "query": (
            "I need something good for photography."
        )
    },

    # -----------------------------------------
    # Negative Queries
    # -----------------------------------------

    {
        "test_id": 13,
        "category": "Negative",
        "query": (
            "Recommend an Apple phone with stylus support."
        )
    },

    {
        "test_id": 14,
        "category": "Negative",
        "query": (
            "Recommend a Samsung phone with a removable battery "
            "and an excellent camera."
        )
    },

    # -----------------------------------------
    # No Match Query
    # -----------------------------------------

    {
        "test_id": 15,
        "category": "No Match",
        "query": (
            "Recommend a Nokia phone with an 8K camera "
            "and 1TB storage."
        )
    }

]

In [20]:
test_case = test_queries[0]

In [21]:
test_results = []

for test_case in test_queries:

    print("=" * 100)
    print(
        f"TEST {test_case['test_id']} "
        f"| {test_case['category']}"
    )
    print("=" * 100)

    try:

        result = (
            run_multi_agent_without_coordination(
                user_query=(
                    test_case["query"]
                ),
                retrieval_k=10,
                max_reviews=3,
                verbose=False
            )
        )

        structured_query = result.get(
            "structured_query"
        )

        ranking_output = result.get(
            "ranking_output"
        )

        ranked_products = (
                ranking_output.products
                if ranking_output is not None
                else []
            )

        top_5_asins = [
                product.parent_asin
                for product in ranked_products[:5]
                if product.parent_asin
            ]

        top_5_titles = [
                product.title
                for product in ranked_products[:5]
                if product.title
            ]

        ranked_count = (
            len(ranking_output.products)
            if ranking_output is not None
            else 0
        )

        test_results.append({
            "test_id": test_case["test_id"],
            "category": test_case["category"],
            "user_query": test_case["query"],

            "product_type": getattr(
                structured_query,
                "product_type",
                None
            ),

            "brand": getattr(
                structured_query,
                "brand",
                None
            ),

            "budget": getattr(
                structured_query,
                "budget",
                None
            ),

            "features": ", ".join(
                getattr(
                    structured_query,
                    "features",
                    []
                ) or []
            ),

            "retrieved_count": len(
                result.get(
                    "retrieved_products",
                    []
                )
            ),

            "ranked_count": ranked_count,

            "recommended_product": (
                result.get(
                    "recommended_product"
                )
            ),

            "recommended_asin": (
                result.get(
                    "recommended_asin"
                )
            ),

            "query_latency": (
                result["latency"]
                ["query_agent"]
            ),

            "retrieval_latency": (
                result["latency"]
                ["retrieval_agent"]
            ),

            "review_latency": (
                result["latency"]
                ["review_agent"]
            ),

            "ranking_latency": (
                result["latency"]
                ["ranking_agent"]
            ),

            "total_latency": (
                result["latency"]
                ["total"]
            ),

            "top_5_asins": "|".join(
                top_5_asins
            ),

            "top_5_titles": " || ".join(
                top_5_titles
            ),


            "error": result.get(
                "error"
            )
        })

        print(
            "Recommended:",
            result.get(
                "recommended_product"
            )
        )

        print(
            "Latency:",
            result["latency"]["total"]
        )

    except Exception as error:

        print(
            "Test failed:",
            type(error).__name__,
            str(error)
        )

        test_results.append({
            "test_id": test_case["test_id"],
            "category": test_case["category"],
            "user_query": test_case["query"],
            "product_type": None,
            "brand": None,
            "budget": None,
            "features": None,
            "retrieved_count": 0,
            "ranked_count": 0,
            "recommended_product": None,
            "recommended_asin": None,
            "query_latency": 0,
            "retrieval_latency": 0,
            "review_latency": 0,
            "ranking_latency": 0,
            "total_latency": 0,
            "top_5_asins": "",
            "top_5_titles": "",
            "error": str(error)
        })

TEST 1 | Brand + Features
Recommended: Samsung Galaxy S21 FE 5G Cell Phone, Factory Unlocked Android Smartphone, 128GB, 120Hz Display, Pro Grade Camera, All Day Intelligent Battery, (Olive Green) (Renewed)
Latency: 3.904
TEST 2 | Brand + Features
Recommended: SteadyMate SM1 HD Professional Handheld Camera Stabilizer for Apple iPhone 6 Plus, 6, 5S, 5C, 5, 4S and 4 Smart Phones
Latency: 2.5443
TEST 3 | Brand + Features
Recommended: Google Pixel XL 128GB - 5.5" Android GSM 4G LTE (GSM Only, No CDMA) Factory Unlocked - International Version - Very Silver
Latency: 3.1721
TEST 4 | Budget
Recommended: Samsung Galaxy A52 (5G) 128GB A526U 6.5" Display Quad Camera Smartphone - Black (Renewed) (AT&T Unlocked)
Latency: 3.2787
TEST 5 | Budget
Recommended: KXD 6A | Unlocked Cell Phone | 5.5" Full Screen Display | 8GB ROM | 2500mAh Battery | 8MP Camera | Android Smartphone | US Version | Black
Latency: 3.493
TEST 6 | Feature Only
Recommended: POSH Revel S500a - 5.0", 4G, Android 4.4 Kit Kat, Dual-cor

In [22]:
no_coordination_results_df = pd.DataFrame(
    test_results
)

display(
    no_coordination_results_df
)

,test_id,category,user_query,product_type,brand,budget,features,retrieved_count,ranked_count,recommended_product,recommended_asin,query_latency,retrieval_latency,review_latency,ranking_latency,total_latency,top_5_asins,top_5_titles,error
0,1,Brand + Features,Recommend a Samsung phone with a good camera a...,smartphone,Samsung,NaN,"good camera, long battery life, camera",8,8,"Samsung Galaxy S21 FE 5G Cell Phone, Factory U...",B0B1QXHB7L,1.0280,0.3640,0.0001,2.5119,3.9040,B0B1QXHB7L|B09S6VKCLX|B09WT8N5X7|B00D93LOY6|B0...,"Samsung Galaxy S21 FE 5G Cell Phone, Factory U...",None
1,2,Brand + Features,I want an Apple phone with excellent camera qu...,smartphone,Apple,NaN,"excellent camera quality, camera",1,1,SteadyMate SM1 HD Professional Handheld Camera...,B00ORZVW42,0.9112,0.3121,0.0000,1.3208,2.5443,B00ORZVW42,SteadyMate SM1 HD Professional Handheld Camera...,None
2,3,Brand + Features,Suggest a Google phone with fast performance.,smartphone,Google,NaN,fast performance,2,2,"Google Pixel XL 128GB - 5.5"" Android GSM 4G LT...",B01M27MVQI,0.8175,0.3683,0.0001,1.9863,3.1721,B01M27MVQI|B08BXBT8MD,"Google Pixel XL 128GB - 5.5"" Android GSM 4G LT...",None
3,4,Budget,Recommend a Samsung phone under 300 with a goo...,smartphone,Samsung,300.0,"good camera, camera",10,10,"Samsung Galaxy A52 (5G) 128GB A526U 6.5"" Displ...",B09C6N8P6Y,0.9225,0.3574,0.0002,1.9987,3.2787,B09C6N8P6Y|B00CGIULGC|B0B1QXHB7L|B00D93LOY6|B0...,"Samsung Galaxy A52 (5G) 128GB A526U 6.5"" Displ...",None
4,5,Budget,Recommend a smartphone under 250 with long bat...,smartphone,None,250.0,long battery life,10,10,"KXD 6A | Unlocked Cell Phone | 5.5"" Full Scree...",B0B74J524C,1.0230,0.4736,0.0001,1.9963,3.4930,B0B74J524C|B07K9PR1H5|B0995SX8X8|B09L4QQCN2|B0...,"KXD 6A | Unlocked Cell Phone | 5.5"" Full Scree...",None
5,6,Feature Only,Recommend a phone with a great camera.,smartphone,None,NaN,"great camera, camera",10,10,"POSH Revel S500a - 5.0"", 4G, Android 4.4 Kit K...",B00O3PAN1E,1.0070,0.4444,0.0001,2.6445,4.0960,B00O3PAN1E|B00NEEXCTK|B00CGIULGC|B002VRO83K|B0...,"POSH Revel S500a - 5.0"", 4G, Android 4.4 Kit K...",None
6,7,Feature Only,Recommend a phone with long battery life.,smartphone,None,NaN,long battery life,10,10,"OUKITEL OK6000 Plus Unlocked Smartphones, 5.5i...",B07K9PR1H5,0.9038,0.3915,0.0002,1.7646,3.0601,B07K9PR1H5|B0995SX8X8|B09L4QQCN2|B01LZ8516T|B0...,"OUKITEL OK6000 Plus Unlocked Smartphones, 5.5i...",None
7,8,Feature Only,Suggest a smartphone for gaming.,smartphone,None,NaN,gaming,10,10,"Black Shark 4 Unlocked Phone, 5G Gaming Phone,...",B09L4QQCN2,1.0339,0.3394,0.0001,2.3239,3.6973,B09L4QQCN2|B0BLHCXZ8K|B00O3PAN1E|B00NEEXCTK|B0...,"Black Shark 4 Unlocked Phone, 5G Gaming Phone,...",None
8,9,Multiple Features,"Recommend a Samsung phone with a good camera, ...",smartphone,Samsung,NaN,"good camera, long battery life, fast performan...",10,10,Samsung Galaxy Mega 6.3 I9200 8GB Unlocked GSM...,B00D93LOY6,1.2221,0.3369,0.0001,1.7072,3.2663,B00D93LOY6|B00JKSUHLU|B0B1QXHB7L|B09S6VKCLX|B0...,Samsung Galaxy Mega 6.3 I9200 8GB Unlocked GSM...,None
9,10,Multiple Features,Recommend an Apple phone with a good camera an...,smartphone,Apple,NaN,"good camera, large storage, camera",3,3,Apple iPhone 5 - 16GB (Black) Factory Unlocked,B00CX0OZHY,1.4370,0.3325,0.0001,3.0380,4.8076,B00CX0OZHY|B00BUYRQG6|B00ORZVW42,Apple iPhone 5 - 16GB (Black) Factory Unlocked...,None


In [23]:
summary_columns = [
    "test_id",
    "category",
    "user_query",
    "product_type",
    "brand",
    "budget",
    "features",

    "retrieved_count",
    "ranked_count",

    "recommended_product",
    "recommended_asin",

    "top_5_asins",
    "top_5_titles",

    "total_latency",
    "error"
]

display(
    no_coordination_results_df[
        summary_columns
    ]
)

,test_id,category,user_query,product_type,brand,budget,features,retrieved_count,ranked_count,recommended_product,recommended_asin,top_5_asins,top_5_titles,total_latency,error
0,1,Brand + Features,Recommend a Samsung phone with a good camera a...,smartphone,Samsung,NaN,"good camera, long battery life, camera",8,8,"Samsung Galaxy S21 FE 5G Cell Phone, Factory U...",B0B1QXHB7L,B0B1QXHB7L|B09S6VKCLX|B09WT8N5X7|B00D93LOY6|B0...,"Samsung Galaxy S21 FE 5G Cell Phone, Factory U...",3.9040,None
1,2,Brand + Features,I want an Apple phone with excellent camera qu...,smartphone,Apple,NaN,"excellent camera quality, camera",1,1,SteadyMate SM1 HD Professional Handheld Camera...,B00ORZVW42,B00ORZVW42,SteadyMate SM1 HD Professional Handheld Camera...,2.5443,None
2,3,Brand + Features,Suggest a Google phone with fast performance.,smartphone,Google,NaN,fast performance,2,2,"Google Pixel XL 128GB - 5.5"" Android GSM 4G LT...",B01M27MVQI,B01M27MVQI|B08BXBT8MD,"Google Pixel XL 128GB - 5.5"" Android GSM 4G LT...",3.1721,None
3,4,Budget,Recommend a Samsung phone under 300 with a goo...,smartphone,Samsung,300.0,"good camera, camera",10,10,"Samsung Galaxy A52 (5G) 128GB A526U 6.5"" Displ...",B09C6N8P6Y,B09C6N8P6Y|B00CGIULGC|B0B1QXHB7L|B00D93LOY6|B0...,"Samsung Galaxy A52 (5G) 128GB A526U 6.5"" Displ...",3.2787,None
4,5,Budget,Recommend a smartphone under 250 with long bat...,smartphone,None,250.0,long battery life,10,10,"KXD 6A | Unlocked Cell Phone | 5.5"" Full Scree...",B0B74J524C,B0B74J524C|B07K9PR1H5|B0995SX8X8|B09L4QQCN2|B0...,"KXD 6A | Unlocked Cell Phone | 5.5"" Full Scree...",3.4930,None
5,6,Feature Only,Recommend a phone with a great camera.,smartphone,None,NaN,"great camera, camera",10,10,"POSH Revel S500a - 5.0"", 4G, Android 4.4 Kit K...",B00O3PAN1E,B00O3PAN1E|B00NEEXCTK|B00CGIULGC|B002VRO83K|B0...,"POSH Revel S500a - 5.0"", 4G, Android 4.4 Kit K...",4.0960,None
6,7,Feature Only,Recommend a phone with long battery life.,smartphone,None,NaN,long battery life,10,10,"OUKITEL OK6000 Plus Unlocked Smartphones, 5.5i...",B07K9PR1H5,B07K9PR1H5|B0995SX8X8|B09L4QQCN2|B01LZ8516T|B0...,"OUKITEL OK6000 Plus Unlocked Smartphones, 5.5i...",3.0601,None
7,8,Feature Only,Suggest a smartphone for gaming.,smartphone,None,NaN,gaming,10,10,"Black Shark 4 Unlocked Phone, 5G Gaming Phone,...",B09L4QQCN2,B09L4QQCN2|B0BLHCXZ8K|B00O3PAN1E|B00NEEXCTK|B0...,"Black Shark 4 Unlocked Phone, 5G Gaming Phone,...",3.6973,None
8,9,Multiple Features,"Recommend a Samsung phone with a good camera, ...",smartphone,Samsung,NaN,"good camera, long battery life, fast performan...",10,10,Samsung Galaxy Mega 6.3 I9200 8GB Unlocked GSM...,B00D93LOY6,B00D93LOY6|B00JKSUHLU|B0B1QXHB7L|B09S6VKCLX|B0...,Samsung Galaxy Mega 6.3 I9200 8GB Unlocked GSM...,3.2663,None
9,10,Multiple Features,Recommend an Apple phone with a good camera an...,smartphone,Apple,NaN,"good camera, large storage, camera",3,3,Apple iPhone 5 - 16GB (Black) Factory Unlocked,B00CX0OZHY,B00CX0OZHY|B00BUYRQG6|B00ORZVW42,Apple iPhone 5 - 16GB (Black) Factory Unlocked...,4.8076,None


In [24]:
results_folder = (
    PROJECT_ROOT
    / "Results"
)

results_folder.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    results_folder
    / "multi_agent_without_coordination_results.csv"
)

no_coordination_results_df.to_csv(
    output_path,
    index=False
)

print(
    "Results saved to:",
    output_path
)

Results saved to: C:\Users\srush\Desktop\Multi agent coordination\Results\multi_agent_without_coordination_results.csv


In [26]:
# =====================================================
# DEFINE SCALABILITY QUERY SET
# =====================================================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()

FINAL_FOLDER = (
    PROJECT_ROOT
    / "Results"
    / "Final"
)

benchmark = pd.read_csv(
    FINAL_FOLDER
    / "final_60_query_benchmark.csv"
)

SCALABILITY_QUERY_IDS = [
    6,
    14,
    21,
    29,
    34,
    37,
    48,
    54
]

scalability_queries = (
    benchmark[
        benchmark[
            "test_id"
        ].isin(
            SCALABILITY_QUERY_IDS
        )
    ]
    .copy()
)

print(
    "Scalability queries:",
    len(scalability_queries)
)

display(
    scalability_queries[
        [
            "test_id",
            "category",
            "user_query"
        ]
    ]
)

Scalability queries: 8


,test_id,category,user_query
5,6,Brand + Feature,Recommend a Samsung phone with a good camera.
13,14,Budget,Recommend a smartphone under 200.
20,21,Feature Only,Recommend a phone with a great camera.
28,29,Multiple Features,Recommend a phone with a good camera and long ...
33,34,Multiple Features,"Recommend a smartphone with a good camera, lon..."
36,37,Complex,Recommend a Samsung phone under 300 with a goo...
47,48,Unsupported,Recommend an Apple phone with stylus support.
53,54,No Match,Recommend a Nokia phone with an 8K camera and ...


In [27]:
from langchain_community.callbacks.manager import (
    get_openai_callback
)


scalability_rows = []


for k in [5, 10, 20, 30]:

    for _, row in scalability_queries.iterrows():

        with get_openai_callback() as cb:

            result = (
                run_multi_agent_without_coordination(
                    user_query=row[
                        "user_query"
                    ],
                    retrieval_k=k,
                    max_reviews=3,
                    verbose=False
                )
            )


        scalability_rows.append({

            "system":
                "Multi-Agent Without Coordination",

            "test_id":
                row[
                    "test_id"
                ],

            "retrieval_k":
                k,

            "latency":
                result[
                    "latency"
                ][
                    "total"
                ],

            "tokens":
                cb.total_tokens
        })


no_coord_scalability = pd.DataFrame(
    scalability_rows
)


no_coord_scalability.to_csv(
    FINAL_FOLDER
    / "no_coord_scalability.csv",
    index=False
)

In [28]:
no_coord_scalability.groupby(
    "retrieval_k"
)[
    [
        "latency",
        "tokens"
    ]
].mean()

,latency,tokens
retrieval_k,,
5,3.987600,1654.250
10,3.497663,1644.125
20,5.630175,2017.250
30,5.169837,2000.750
